<a href="https://colab.research.google.com/github/eemarting/Fundamentos-de-Programacion-Python/blob/main/Sesion4IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 4 — Interfaces, Deployment y Proyecto Integrador

**Módulo: Python para IA** | Máster en Inteligencia Artificial

## Objetivos de esta sesión

- Conocer conceptos de **Python avanzado** necesarios para leer y generar código profesional.
- Crear **interfaces web** para modelos de IA con **Gradio**.
- Construir **aplicaciones de datos** con **Streamlit**.
- **Desplegar** aplicaciones en HuggingFace Spaces y Streamlit Cloud.
- Presentar el **proyecto final integrador**.

> 🎯 Esta es la sesión donde todo se junta: del código en un notebook a una **aplicación web desplegada** que cualquier persona puede usar.

### ¿Por qué esta sesión es clave?

A lo largo de las tres sesiones anteriores habéis adquirido herramientas para **escribir Python**, **manipular datos**, **consumir modelos de IA** y **construir sistemas RAG**. Sin embargo, todo ese trabajo vivía dentro de un notebook — accesible solo para vosotros.

En el mundo profesional, un modelo de IA solo tiene valor cuando **alguien puede usarlo**. Esto implica:

1. **Interfaces**: Que un usuario sin conocimientos técnicos pueda interactuar con el modelo (escribir texto, subir archivos, ver resultados).
2. **Deployment**: Que la aplicación esté accesible 24/7 en una URL pública, sin necesidad de abrir un notebook.
3. **Código profesional**: Que el código sea legible, mantenible y siga buenas prácticas (excepciones, tipado, documentación).

Esta sesión cierra el ciclo completo: **idea → código → modelo → interfaz → despliegue**. Al terminar, seréis capaces de construir y publicar una aplicación web con IA de principio a fin.

### Mapa de la sesión

| Bloque | Tema | Duración | Resultado |
|--------|------|:--------:|-----------|
| 1 | Python Avanzado Express | 15 min | Leer/generar código profesional |
| 2 | Gradio | 20 min | Interfaz web para un modelo IA |
| 3 | Streamlit | 15 min | App de datos completa |
| 4 | Deployment | 15 min | App desplegada online |
| 5 | Proyecto Final | 15 min | Definición del entregable |

---
# Bloque 1 — Python Avanzado Express (15 min)

Conceptos que necesitáis conocer para **leer código profesional y pedirlo a la IA**.

### ¿Por qué "Python Avanzado" en un módulo de IA?

Cuando trabajáis con librerías del ecosistema IA (HuggingFace, LangChain, Gradio, etc.), el código que encontráis en documentación, tutoriales y ejemplos usa constantemente conceptos "avanzados" de Python: **clases**, **decoradores**, **manejo de excepciones**, **type hints**...

El objetivo de este bloque **no es que os convirtáis en expertos en POO** (eso lo haréis en otros módulos). El objetivo es que:

- **Reconozcáis** estos patrones cuando los veáis en código de librerías IA.
- **Sepáis pedírselos a la IA** cuando generéis código profesional.
- **Entendáis** por qué el código generado por Copilot/Gemini tiene esa estructura.

> 🤖 **Prompting tip**: Cuando le pidas código a Gemini/Copilot, puedes especificar el nivel de complejidad: *"Genera este código usando clases con @dataclass y manejo de excepciones"* vs *"Genera esto como una función simple"*. Saber que estos conceptos existen os da **vocabulario** para pedir mejor código.

## Excepciones — Manejo de Errores

### ¿Por qué es imprescindible en aplicaciones IA?

En un notebook, si algo falla, simplemente veis un error rojo y corregís el código. Pero en una **aplicación desplegada** (como las que vamos a crear con Gradio/Streamlit), un error no manejado **rompe la app** para el usuario. Las excepciones permiten que tu programa **detecte el error, reaccione de forma controlada y siga funcionando**.

Esto es especialmente crítico en IA porque las fuentes de error son muchas:
- **APIs externas** que fallan o están caídas (Gemini, OpenAI, HuggingFace).
- **Límites de uso** alcanzados (rate limits, cuotas).
- **Datos inesperados** del usuario (texto vacío, archivos corruptos, idiomas no soportados).
- **Modelos** que generan respuestas inesperadas o vacías.

### Estructura básica: `try` / `except` / `finally`

```python
try:
    # Código que PUEDE fallar
    resultado = 10 / 0
except ZeroDivisionError:
    # Se ejecuta SOLO si ocurre ese error específico
    print("No se puede dividir por cero")
except Exception as e:
    # Se ejecuta para CUALQUIER otro error
    print(f"Error inesperado: {e}")
finally:
    # Se ejecuta SIEMPRE, haya error o no
    # Útil para cerrar conexiones, liberar recursos, etc.
    print("Esto siempre se ejecuta")
```

### Tipos de excepciones más comunes en IA

| Excepción | Cuándo ocurre | Ejemplo típico |
|-----------|--------------|----------------|
| `ValueError` | Valor incorrecto | Texto vacío pasado a un modelo |
| `TypeError` | Tipo incorrecto | Pasar un int donde se espera str |
| `KeyError` | Clave no existe en dict | Acceder a `response["choices"]` cuando la API devuelve error |
| `FileNotFoundError` | Archivo no existe | Cargar un modelo local que no se descargó |
| `ConnectionError` | Fallo de red | API de Gemini no accesible |
| `TimeoutError` | Operación tardó demasiado | Modelo muy grande en CPU |

### Patrón común en aplicaciones IA:
```python
try:
    respuesta = modelo.generate(prompt)
except APIError as e:
    respuesta = "Lo siento, el servicio no está disponible"
    log_error(e)
```

> 🤖 **Prompting tip**: Cuando pidas código a la IA para una app, incluye *"con manejo de excepciones para errores de API y datos inválidos"*. Esto hará que el código generado sea robusto desde el principio.

In [ ]:
# Excepciones en la práctica
def dividir_seguro(a: float, b: float) -> float | str:
    """Divide a/b de forma segura."""
    try:
        return a / b
    except ZeroDivisionError:
        return "Error: división por cero"
    except TypeError:
        return "Error: tipos no válidos"

print(dividir_seguro(10, 3))     # 3.333...
print(dividir_seguro(10, 0))     # Error: división por cero
print(dividir_seguro("a", 3))   # Error: tipos no válidos

3.3333333333333335
Error: división por cero
Error: tipos no válidos


## POO en Alto Nivel — Clases y Decoradores

### ¿Por qué aparecen clases por todas partes en IA?

Prácticamente **todas las librerías del ecosistema IA** están construidas con Programación Orientada a Objetos (POO). Cuando hacéis `pipeline("sentiment-analysis")`, por detrás se crea un **objeto** con métodos y atributos. Cuando usáis `genai.GenerativeModel("gemini-2.0-flash")`, estáis **instanciando una clase**.

No necesitáis dominar POO para este módulo, pero sí entender la mecánica básica para:
- **Leer código** de documentación y ejemplos (que usa clases constantemente).
- **Pedir a la IA** que genere código con una estructura organizada.
- **Extender** funcionalidad de librerías existentes (algo habitual en proyectos reales).

### Clases: lo esencial

Una **clase** es un molde para crear objetos. El objeto tiene **atributos** (datos) y **métodos** (acciones):

```python
class MiModelo:
    def __init__(self, nombre: str):      # Constructor: se ejecuta al crear el objeto
        self.nombre = nombre               # Atributo público
        self._historial = []               # Convención: _ = "privado" (no tocar desde fuera)

    def predecir(self, texto: str) -> str: # Método: acción que el objeto puede hacer
        resultado = f"Predicción para '{texto}'"
        self._historial.append(resultado)
        return resultado
```

**Lectura del código**: `self` es una referencia al propio objeto. Siempre es el primer parámetro de los métodos, pero no se pasa al llamarlos: `modelo.predecir("hola")` (no `modelo.predecir(modelo, "hola")`).

### Decoradores importantes

Los **decoradores** son modificadores que se colocan encima de una función o método con `@` y cambian su comportamiento. Los veréis constantemente en código IA:

| Decorador | Qué hace | Ejemplo en IA |
|-----------|----------|---------------|
| `@property` | Acceder a un método como atributo (`obj.nombre` en vez de `obj.nombre()`) | `modelo.config` en HuggingFace |
| `@staticmethod` | Método que no necesita `self` (como una función normal dentro de la clase) | Funciones de utilidad |
| `@classmethod` | Método que recibe la clase, útil para crear objetos de formas alternativas | `Model.from_pretrained()` en HuggingFace |
| `@dataclass` | Genera `__init__`, `__repr__`, `__eq__` automáticamente — menos código repetitivo | Configuraciones de modelos |

### `@dataclass` — El decorador que más usaréis

Las **dataclasses** son clases donde lo importante son los datos, no los métodos. Python genera automáticamente el constructor, la representación en texto y la comparación. Es la forma más limpia de agrupar datos relacionados:

```python
@dataclass
class ConfigModelo:
    nombre: str
    temperatura: float = 0.7
    max_tokens: int = 1000
```

Sin `@dataclass`, tendríais que escribir manualmente `__init__`, `__repr__` y `__eq__` — unas 15 líneas extra de código repetitivo.

> 🤖 **Prompting tip**: Si necesitáis agrupar configuraciones o datos, pedid a la IA: *"Crea una dataclass para representar la configuración de mi modelo con nombre, temperatura y max_tokens"*. Es mucho más limpio que usar diccionarios.

In [ ]:
from dataclasses import dataclass

# dataclass genera automáticamente __init__, __repr__, __eq__
@dataclass
class Producto:
    nombre: str
    precio: float
    stock: int = 0

    @property
    def valor_total(self) -> float:
        """Valor total = precio × stock"""
        return self.precio * self.stock

    @property
    def disponible(self) -> bool:
        return self.stock > 0

# Uso
laptop = Producto("Laptop", 999.99, 5)
mouse = Producto("Mouse", 29.99, 50)

print(laptop)                     # Producto(nombre='Laptop', precio=999.99, stock=5)
print(f"Valor total: {laptop.valor_total}€")    # Accede como atributo
print(f"Disponible: {laptop.disponible}")

# Comparar
print(f"laptop == mouse: {laptop == mouse}")
print(f"laptop == Producto('Laptop', 999.99, 5): {laptop == Producto('Laptop', 999.99, 5)}")

Producto(nombre='Laptop', precio=999.99, stock=5)
Valor total: 4999.95€
Disponible: True
laptop == mouse: False
laptop == Producto('Laptop', 999.99, 5): True


## Type Hints y Docstrings

### Type Hints — Decir qué tipo de datos esperas

Los **type hints** (anotaciones de tipo) indican qué tipo de dato recibe y devuelve una función. Python **no los obliga** (no da error si pasas un tipo diferente), pero aportan enormes beneficios:

- **Legibilidad**: Cualquier persona (o IA) que lea el código entiende inmediatamente qué entra y qué sale.
- **Autocompletado**: Los editores como VS Code usan los type hints para sugerir métodos y detectar errores.
- **Documentación viva**: Los tipos actúan como documentación que nunca queda desactualizada.

```python
# Sin type hints — ¿qué recibe? ¿qué devuelve? Hay que leer todo el código para saberlo
def procesar(datos, filtro):
    ...

# Con type hints — inmediatamente claro
def procesar(datos: list[dict], filtro: str = "todos") -> pd.DataFrame:
    ...
```

### Docstrings — Documentar funciones y clases

Las **docstrings** son cadenas de texto al inicio de una función/clase que explican qué hace, qué recibe y qué devuelve. El estilo más usado en el ecosistema Google/IA es el **estilo Google**:

```python
def procesar_datos(datos: list[dict], filtro: str = "todos") -> pd.DataFrame:
    """Procesa una lista de datos y devuelve un DataFrame filtrado.

    Args:
        datos: Lista de diccionarios con los datos a procesar.
        filtro: Criterio de filtrado. Por defecto "todos".

    Returns:
        DataFrame con los datos procesados y filtrados.

    Raises:
        ValueError: Si la lista de datos está vacía.
    """
```

### ¿Por qué esto importa para vosotros?

Cuando generáis código con IA, el código con type hints y docstrings es:
- **Más fácil de entender** por vosotros (sabéis qué hace cada función sin ejecutarla).
- **Más fácil de modificar** por la IA en iteraciones posteriores (tiene contexto sobre tipos).
- **Más profesional** — es lo que se espera en código de producción.

> 💡 **Pro tip**: Cuando le pidas código a la IA, incluye siempre *"con type hints y docstrings estilo Google"* en tu prompt. El código será mucho más fácil de entender y mantener. Este es un ejemplo perfecto de cómo saber que algo existe os da poder para **pedir mejor código**.

---
# Bloque 2 — Gradio: Interfaces para Modelos IA (20 min)

Hasta ahora, vuestros modelos y pipelines se ejecutan dentro de un notebook — solo vosotros podéis verlos. **Gradio** permite convertir cualquier función Python en una **aplicación web interactiva** en cuestión de minutos. Es la herramienta estándar de la comunidad IA para crear demos y prototipos.

Si habéis explorado modelos en HuggingFace Hub, probablemente ya habéis usado Gradio sin saberlo: la mayoría de las **demos interactivas** que aparecen en las páginas de modelos están construidas con Gradio.

## ¿Qué es Gradio?

[Gradio](https://gradio.app/) es una librería Python (propiedad de HuggingFace) que permite crear **interfaces web** para cualquier función Python en pocas líneas de código. Su filosofía es simple: si tienes una función que recibe datos y devuelve resultados, Gradio la convierte en una app web.

### ¿Cómo funciona conceptualmente?

```
Tu función Python                    Gradio genera automáticamente
─────────────────                    ──────────────────────────────
def analizar(texto: str) -> str:     ┌──────────────────────┐
    ...                         ──►  │ [Campo de texto]     │
    return resultado                 │ [Botón: Submit]      │
                                     │ [Resultado]          │
                                     └──────────────────────┘
```

Gradio se encarga de toda la parte web (HTML, CSS, JavaScript, servidor) — vosotros solo escribís la **función Python**.

### ¿Por qué Gradio?
- ✅ Perfecto para **demos de modelos ML/IA** — es el estándar de facto en HuggingFace
- ✅ Se ejecuta **dentro de Google Colab** sin configuración extra
- ✅ Se despliega fácilmente en **HuggingFace Spaces** (gratis, incluso con GPU)
- ✅ Soporta múltiples tipos de datos: texto, imágenes, audio, video, archivos, DataFrames...
- ✅ Incluye **`gr.ChatInterface`** para crear chatbots en 5 líneas de código

### Dos formas de crear interfaces

| Forma | Cuándo usarla | Complejidad |
|-------|--------------|:-----------:|
| **`gr.Interface`** | Interfaces simples: input → función → output | ⭐ |
| **`gr.Blocks`** | Interfaces complejas: layouts personalizados, múltiples componentes, eventos | ⭐⭐ |

La mayoría de demos se pueden hacer con `gr.Interface`. Usad `gr.Blocks` cuando necesitéis más control (chatbots, múltiples secciones, pestañas, etc.).

### Gradio vs Streamlit — Primera comparativa

| Aspecto | Gradio | Streamlit |
|---------|--------|-----------|
| **Ideal para** | Demos de modelos ML | Apps de datos completas |
| **Interfaz** | Input → Output (funcional) | Layout libre (imperativo) |
| **En Colab** | ✅ Nativo | ⚠️ Via tunnel (más complejo) |
| **Deployment** | HuggingFace Spaces | Streamlit Cloud |
| **Curva de aprendizaje** | Muy baja | Baja |
| **Ecosistema** | HuggingFace | General / GitHub |

> 🤖 **Prompting tip**: Cuando pidáis a la IA crear una interfaz Gradio, especificad: *"Usa gr.Interface para una demo simple"* o *"Usa gr.Blocks si necesito layout personalizado con pestañas"*. Esto evita que la IA genere código más complejo de lo necesario.

In [ ]:
# !pip install gradio -q
import gradio as gr

# Ejemplo 1: Interfaz básica — una función con input y output
def saludar(nombre: str, idioma: str) -> str:
    saludos = {
        "Español": f"¡Hola, {nombre}! 👋",
        "English": f"Hello, {nombre}! 👋",
        "Français": f"Bonjour, {nombre}! 👋",
    }
    return saludos.get(idioma, f"Hi, {nombre}!")

# Crear interfaz
demo = gr.Interface(
    fn=saludar,
    inputs=[
        gr.Textbox(label="Tu nombre", placeholder="Escribe tu nombre..."),
        gr.Dropdown(choices=["Español", "English", "Français"], label="Idioma", value="Español"),
    ],
    outputs=gr.Textbox(label="Saludo"),
    title="🌍 Saludador Multiidioma",
    description="Escribe tu nombre y selecciona un idioma para recibir un saludo personalizado."
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://567e87c2ac4570daeb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# Ejemplo 2: Interfaz con modelo de HuggingFace
from transformers import pipeline

# Cargar modelo de sentimiento
clasificador = pipeline("sentiment-analysis")

def analizar_sentimiento(texto: str) -> dict:
    """Analiza el sentimiento de un texto."""
    resultado = clasificador(texto)[0]
    return {
        "POSITIVE 😊": resultado["score"] if resultado["label"] == "POSITIVE" else 1 - resultado["score"],
        "NEGATIVE 😞": resultado["score"] if resultado["label"] == "NEGATIVE" else 1 - resultado["score"],
    }

demo_sentimiento = gr.Interface(
    fn=analizar_sentimiento,
    inputs=gr.Textbox(
        label="Texto a analizar",
        placeholder="Escribe una reseña, opinión o comentario...",
        lines=3
    ),
    outputs=gr.Label(label="Sentimiento"),
    title="🎭 Análisis de Sentimiento",
    description="Escribe un texto y el modelo determinará si es positivo o negativo.",
    examples=[
        ["Me encanta este producto, funciona genial!"],
        ["Terrible experiencia, no lo recomiendo para nada"],
        ["Está bien, funciona pero nada especial"],
    ]
)

demo_sentimiento.launch()

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7de4322add6f5e9c76.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# Ejemplo 3: Chatbot con Gemini
import google.generativeai as genai
import gradio as gr
from google.colab import userdata

try:
    genai.configure(api_key=userdata.get("GEMINI_API_KEY"))
except Exception as e:
    print(f"Advertencia: No se pudo configurar la API Key. {e}")

def chatbot(mensaje: str, historial: list):
    """Chatbot con Gemini que mantiene historial."""
    try:
        model = genai.GenerativeModel("gemini-2.5-flash")

        # Construir contexto del historial
        contexto = "Eres un asistente amable y conciso. "
        for h_msg, h_resp in historial:
            contexto += f"Usuario: {h_msg}\nAsistente: {h_resp}\n"
        contexto += f"Usuario: {mensaje}\nAsistente:"

        response = model.generate_content(contexto)
        respuesta = response.text
    except Exception as e:
        respuesta = f"Error: {e}. Asegúrate de configurar la API Key de Gemini en los Secretos de Colab."

    historial.append((mensaje, respuesta))
    return "", historial

# Interfaz de chat usando gr.Blocks
with gr.Blocks(title="💬 Chat con Gemini") as demo_chat:
    gr.Markdown("# 💬 Chat con Gemini\nConversa con el modelo Gemini de Google.")

    chatbot_ui = gr.Chatbot(height=400)
    msg = gr.Textbox(label="Tu mensaje", placeholder="Escribe algo y pulsa Enter...")
    clear = gr.Button("🗑️ Limpiar chat")

    # Eventos
    msg.submit(chatbot, [msg, chatbot_ui], [msg, chatbot_ui])
    clear.click(lambda: (None, []), outputs=[msg, chatbot_ui])

demo_chat.launch(debug=True)

/tmp/ipython-input-2050251005.py:34: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot_ui = gr.Chatbot(height=400)
/tmp/ipython-input-2050251005.py:34: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot_ui = gr.Chatbot(height=400)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8ffea564c60d1a7c8c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7863 <> https://8ffea564c60d1a7c8c.gradio.live


### 🛠️ Ejercicio — Crea tu propia interfaz Gradio

Pide a Gemini:

> *"Crea una interfaz Gradio con gr.Blocks que tenga: un Textbox donde el usuario escriba un tema, un Slider para elegir la longitud (corto/medio/largo), y un botón que al hacer clic llame a Gemini para generar un resumen sobre ese tema con la longitud especificada. Muestra el resultado en un Markdown component."*

In [ ]:
# Pega aquí tu interfaz Gradio personalizada


In [ ]:
import google.generativeai as genai
import gradio as gr
from google.colab import userdata

# Eliminar la configuración global de la API Key para moverla dentro de la función
# try:
#     genai.configure(api_key=userdata.get("GEMINI_API_KEY"))
# except Exception as e:
#     print(f"Advertencia: No se pudo configurar la API Key. {e}")

def generar_resumen(tema: str, longitud: str) -> str:
    """Genera un resumen sobre un tema dado con la longitud especificada usando Gemini."""
    if not tema:
        return "Por favor, introduce un tema para resumir."

    api_key = userdata.get("GEMINI_API_KEY")
    if not api_key:
        return "Error: API Key de Gemini no configurada en los Secretos de Colab. Por favor, añádela para usar esta función."

    try:
        genai.configure(api_key=api_key)
        model = genai.GenerativeModel("gemini-2.5-flash")
        prompt = f"Genera un resumen {longitud} sobre el siguiente tema: {tema}."
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"Error al generar el resumen: {e}. Asegúrate de que tu API Key es válida y que el servicio está disponible."

# Interfaz Gradio con gr.Blocks
with gr.Blocks(title="📝 Generador de Resúmenes con Gemini") as demo_resumen:
    gr.Markdown("# 📝 Generador de Resúmenes con Gemini\nIntroduce un tema y la longitud deseada para obtener un resumen.")

    with gr.Row():
        tema_input = gr.Textbox(label="Tema", placeholder="Ej: La historia de la IA", lines=2)
        longitud_slider = gr.Slider(
            minimum=0, maximum=2, step=1, value=1, label="Longitud del Resumen",
            # Convertir valor numérico a texto para el prompt
            show_label=True,
            interactive=True
        )

    generar_button = gr.Button("Generar Resumen")
    output_resumen = gr.Markdown(label="Resumen Generado")

    def get_longitud_text(value):
        if value == 0:
            return "corto"
        elif value == 1:
            return "medio"
        else:
            return "largo"

    generar_button.click(
        fn=lambda t, l: generar_resumen(t, get_longitud_text(l)),
        inputs=[tema_input, longitud_slider],
        outputs=output_resumen
    )

demo_resumen.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a108b3f60ee6b33fb0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
# Bloque 3 — Streamlit: Apps de Datos e IA (15 min)

Mientras que Gradio está pensado para **demos de modelos** (input → output), **Streamlit** es un framework completo para construir **aplicaciones de datos**: dashboards, herramientas internas, chatbots con interfaz rica, aplicaciones con múltiples páginas, etc.

Streamlit es enormemente popular en la industria de datos/IA porque permite a **data scientists y ML engineers** crear aplicaciones web sin conocimientos de frontend (HTML/CSS/JavaScript). Empresas como Snowflake (que compró Streamlit), Uber y muchas startups de IA lo usan en producción.

## ¿Qué es Streamlit?

[Streamlit](https://streamlit.io/) es un framework Python para crear **aplicaciones web de datos** de forma rápida. A diferencia de Gradio (que se centra en input→output), Streamlit te da **control total sobre el layout** de la página.

### ¿Cómo funciona Streamlit?

El modelo mental de Streamlit es diferente al de un notebook:
1. Escribes un **script Python** (no un notebook) — normalmente `app.py`.
2. Streamlit ejecuta ese script **de arriba a abajo** cada vez que el usuario interactúa.
3. Los componentes `st.xxx()` van apareciendo en la página en el orden en que los llamas.

Esto significa que **no necesitáis HTML ni JavaScript** — todo se construye con llamadas Python:

### Componentes principales:

```python
import streamlit as st

# --- Layout y texto ---
st.title("Mi App")                   # Título grande
st.header("Sección")                 # Subtítulo
st.write("Texto normal")             # Texto / Markdown / DataFrames (mágico: detecta el tipo)
st.markdown("**Negrita** y *cursiva*") # Markdown explícito

# --- Inputs del usuario ---
nombre = st.text_input("Nombre")              # Input de texto → devuelve string
edad = st.slider("Edad", 0, 100)              # Slider → devuelve número
opcion = st.selectbox("Elige", ["A", "B"])    # Dropdown → devuelve selección
archivo = st.file_uploader("Sube un archivo") # Subir archivos
boton = st.button("Click me")                 # Botón → devuelve True/False

# --- Layout avanzado ---
st.sidebar.title("Menú lateral")     # Sidebar (panel lateral)
col1, col2 = st.columns(2)           # Columnas

# --- Chat (para chatbots) ---
st.chat_message("user")              # Mensaje de chat (burbuja)
st.chat_input("Escribe algo...")     # Input de chat (barra inferior)
```

### `st.session_state` — La memoria de Streamlit

Como Streamlit re-ejecuta todo el script en cada interacción, **las variables normales se pierden**. Para mantener estado entre interacciones (historial de chat, datos cargados, etc.), se usa `st.session_state`:

```python
# Inicializar solo la primera vez
if "contador" not in st.session_state:
    st.session_state.contador = 0

# Modificar en cada interacción
if st.button("Incrementar"):
    st.session_state.contador += 1

st.write(f"Contador: {st.session_state.contador}")
```

### Ejecutar Streamlit:
```bash
# En terminal local
pip install streamlit
streamlit run app.py

# En Colab (requiere tunnel — es más complejo que Gradio)
!streamlit run app.py &
```

> ⚠️ **Importante**: Streamlit no se ejecuta nativamente en Colab como Gradio. En esta sesión veremos el código y lo desplegaremos directamente en Streamlit Cloud. Para desarrollo local, ejecutáis `streamlit run app.py` en vuestra terminal.

In [ ]:
# Ejemplo de app Streamlit — guardarla como archivo

app_streamlit = '''
import streamlit as st
import google.generativeai as genai

st.set_page_config(page_title="Chat IA", page_icon="🤖")

st.title("🤖 Chat con Gemini")
st.markdown("Una app sencilla para conversar con Gemini.")

# Sidebar para configuración
with st.sidebar:
    st.header("⚙️ Configuración")
    api_key = st.text_input("API Key de Gemini", type="password")
    temperatura = st.slider("Temperatura", 0.0, 1.0, 0.7)
    st.markdown("---")
    st.markdown("[Obtener API Key](https://aistudio.google.com/apikey)")

# Inicializar historial
if "mensajes" not in st.session_state:
    st.session_state.mensajes = []

# Mostrar historial
for msg in st.session_state.mensajes:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

# Input del usuario
if prompt := st.chat_input("Escribe tu mensaje..."):
    # Mostrar mensaje del usuario
    with st.chat_message("user"):
        st.write(prompt)
    st.session_state.mensajes.append({"role": "user", "content": prompt})

    # Generar respuesta
    if api_key:
        genai.configure(api_key=api_key)
        model = genai.GenerativeModel("gemini-2.0-flash")
        response = model.generate_content(
            prompt,
            generation_config=genai.GenerationConfig(temperature=temperatura)
        )
        respuesta = response.text
    else:
        respuesta = "⚠️ Añade tu API Key de Gemini en la barra lateral."

    # Mostrar respuesta
    with st.chat_message("assistant"):
        st.write(respuesta)
    st.session_state.mensajes.append({"role": "assistant", "content": respuesta})
'''

# Guardar como archivo
with open("app_chat.py", "w") as f:
    f.write(app_streamlit)

print("✅ app_chat.py guardado")
print("\nPara ejecutar localmente:")
print("  pip install streamlit google-generativeai")
print("  streamlit run app_chat.py")

✅ app_chat.py guardado

Para ejecutar localmente:
  pip install streamlit google-generativeai
  streamlit run app_chat.py


### Gradio vs Streamlit — ¿Cuándo usar cada uno?

Esta es la pregunta más frecuente. La respuesta corta: **depende del caso de uso**. La respuesta larga:

| Caso de uso | Recomendación | ¿Por qué? |
|------------|---------------|-----------|
| Demo rápida de un modelo | **Gradio** | Input→Output en 3 líneas |
| App de datos con gráficos y tablas | **Streamlit** | Layout flexible, integración nativa con Pandas/Plotly |
| Dentro de Google Colab | **Gradio** | Funciona nativamente sin tunnels |
| Chat con historial | Ambos | Gradio: `gr.ChatInterface` / Streamlit: `chat_message` |
| Despliegue en HuggingFace | **Gradio** | Integración nativa con HF Spaces |
| Despliegue general | **Streamlit** Cloud | Más flexible, conecta directo con GitHub |
| Interfaz I/O simple | **Gradio** | Menos código, más rápido |
| Layout complejo con sidebar | **Streamlit** | Control total del layout |
| Múltiples páginas | **Streamlit** | Soporte nativo de multi-page apps |
| Compartir con compañeros rápidamente | **Gradio** | `share=True` genera una URL temporal |

### Regla práctica

> **¿Tu app es esencialmente "el usuario mete algo → el modelo devuelve algo"?** → Usa **Gradio**.
>
> **¿Tu app necesita navegación, estado complejo, múltiples secciones, dashboards?** → Usa **Streamlit**.
>
> **¿No estás seguro?** → Empieza con **Gradio** (menos código). Si se queda corto, migra a Streamlit.

Para el **proyecto final**, ambas opciones son igualmente válidas. Elegid la que mejor se adapte a lo que queréis construir.

---
# Bloque 4 — Deployment: Del Notebook a la Web (15 min)

### ¿Qué significa "deployment" (despliegue)?

**Deployment** es el proceso de llevar tu código desde tu máquina local (o un notebook) a un **servidor accesible por internet**. Después del despliegue, cualquier persona con el link puede usar tu aplicación sin instalar nada.

### ¿Por qué es importante?

En IA, un modelo que solo funciona en tu notebook **no tiene impacto real**. El despliegue es lo que transforma un experimento en un **producto**:

| Sin deployment | Con deployment |
|----------|----------|
| Solo tú puedes usarlo | Cualquier persona con el link |
| Necesita Colab/Jupyter abierto | Funciona 24/7 automáticamente |
| "Mira este notebook" | "Prueba esta app: https://mi-app.hf.space" |
| Experimento | Producto / Demo profesional |

### Opciones gratuitas para desplegar apps IA

| Plataforma | Framework | Gratis | GPU gratis | Ideal para |
|-----------|-----------|:------:|:----------:|------------|
| **HuggingFace Spaces** | Gradio / Streamlit | ✅ | ✅ (limitada) | Demos de modelos ML |
| **Streamlit Cloud** | Streamlit | ✅ | ❌ | Apps de datos, chatbots con API |
| **Render** | Cualquiera | ✅ (limitado) | ❌ | Apps generales |
| **Railway** | Cualquiera | ✅ (limitado) | ❌ | Apps generales |

Para este módulo nos centramos en las **dos opciones más simples y gratuitas**: HuggingFace Spaces y Streamlit Cloud.

## Desplegar en HuggingFace Spaces (Gradio)

HuggingFace Spaces permite desplegar apps **gratis** con GPU incluida (limitada). Es la opción más directa si usáis Gradio, porque HuggingFace es la empresa que desarrolla Gradio.

### ¿Cómo funciona por detrás?

Cuando creas un Space, HuggingFace:
1. **Crea un contenedor Docker** con tu código.
2. **Instala las dependencias** de `requirements.txt`.
3. **Ejecuta** `app.py` automáticamente.
4. **Expone** la interfaz en una URL pública: `https://tu-usuario-tu-app.hf.space`.

Cada vez que actualizas el código (vía la web o Git), el Space se **reconstruye automáticamente**.

### Pasos:

1. **Ir a** [huggingface.co/new-space](https://huggingface.co/new-space)
2. **Configurar**:
   - Nombre del Space (será parte de la URL)
   - SDK: **Gradio** (o Streamlit, Docker)
   - Visibilidad: Public (cualquiera puede ver) o Private
   - Hardware: CPU Basic (gratis) o GPU (gratis con límites)
3. **Crear los archivos** — la estructura mínima es:

```
mi-app/
├── app.py              # Tu código Gradio (DEBE tener demo.launch())
└── requirements.txt    # Dependencias (una por línea)
```

4. **`app.py`** — Tu código Gradio (la función + `demo.launch()`):
```python
import gradio as gr
from transformers import pipeline

clasificador = pipeline("sentiment-analysis")

def analizar(texto):
    resultado = clasificador(texto)[0]
    return f"{resultado['label']}: {resultado['score']:.3f}"

demo = gr.Interface(fn=analizar, inputs="text", outputs="text")
demo.launch()
```

5. **`requirements.txt`** — Lista de dependencias (sin versiones a menos que sea necesario):
```
transformers
torch
```

6. **Subir** vía la interfaz web de HuggingFace (más fácil) o con Git:
```bash
git clone https://huggingface.co/spaces/tu-usuario/mi-app
cd mi-app
# (añadir archivos app.py y requirements.txt)
git add . && git commit -m "primera version" && git push
```

### Gestión de secretos (API Keys)

Si tu app usa APIs (como Gemini), **nunca pongas la API key en el código**. En HuggingFace Spaces:
- Ve a **Settings → Repository secrets**
- Añade tu secreto (ej: `GEMINI_API_KEY`)
- En el código, accede con: `os.environ["GEMINI_API_KEY"]`

> ⚠️ **Errores comunes en Spaces**: 1) Olvidar poner una dependencia en `requirements.txt`. 2) Usar una versión incompatible de una librería. 3) No llamar a `demo.launch()` al final de `app.py`. Si el Space falla, revisad los **logs** en la pestaña "Logs" del Space.

## Desplegar en Streamlit Cloud

[Streamlit Cloud](https://streamlit.io/cloud) permite desplegar apps desde un **repositorio de GitHub**. Es la opción natural si usáis Streamlit, y tiene una ventaja importante: **se actualiza automáticamente** cada vez que hacéis push al repo.

### ¿Cómo funciona por detrás?

1. Streamlit Cloud **monitoriza** tu repositorio de GitHub.
2. Cada vez que haces `git push`, **detecta cambios** y re-despliega automáticamente.
3. Tu app se ejecuta en un servidor de Streamlit con una URL pública.

Esto significa que vuestro flujo de trabajo es: **editar código → git push → la app se actualiza sola**. No hay que hacer deploy manual.

### Pasos:

1. **Crear un repo en GitHub** con la estructura mínima:
```
mi-streamlit-app/
├── app.py              # Tu código Streamlit
├── requirements.txt    # Dependencias
└── README.md           # Descripción del proyecto
```

2. **Ir a** [share.streamlit.io](https://share.streamlit.io) y hacer login con GitHub
3. **Conectar tu repositorio de GitHub** (seleccionar repo, rama y archivo)
4. **Seleccionar** el archivo `app.py` como punto de entrada
5. **Deploy** — En unos minutos tu app estará online en `https://tu-app.streamlit.app`

### Para secretos (API Keys):

En Streamlit Cloud, los secretos se gestionan desde el panel web:

1. **En el panel de Streamlit Cloud**: Ve a tu app → Settings → Secrets
2. **Formato TOML** (como un archivo de configuración):
```toml
# Se escribe en el panel de Streamlit Cloud
GEMINI_API_KEY = "tu-api-key-aqui"
```

3. **En el código**, accede con:
```python
import streamlit as st
api_key = st.secrets["GEMINI_API_KEY"]
```

4. **Para desarrollo local**, crea un archivo `.streamlit/secrets.toml` en tu proyecto (y añádelo a `.gitignore` para no subirlo a GitHub):
```toml
# .streamlit/secrets.toml (NO subir a GitHub)
GEMINI_API_KEY = "tu-api-key-aqui"
```

> 💡 **Ventaja de Streamlit Cloud**: Como está conectado a GitHub, podéis usar **Copilot para editar el código** y simplemente hacer push — la app se actualiza sola. Es un flujo de trabajo muy natural para desarrollo agéntico.

### 🛠️ Demo Guiada — Despliegue en HuggingFace Spaces

Vamos a desplegar juntos la app de análisis de sentimiento. El objetivo es que veáis el proceso completo: del código en un notebook a una **URL pública** que cualquiera puede visitar.

### Pasos que seguiremos:

1. **Crear el Space**: Abre [huggingface.co/new-space](https://huggingface.co/new-space)
   - Nombre: `sentiment-analyzer`
   - SDK: Gradio
   - Hardware: CPU Basic (gratis)
2. **Subir `app.py`**: Copia el código del ejemplo de análisis de sentimiento (Bloque 2)
3. **Subir `requirements.txt`**: Con `transformers` y `torch`
4. **Esperar al build**: HuggingFace instalará las dependencias y arrancará la app (~2-3 min)
5. **Probar**: Tu app estará en `https://tu-usuario-sentiment-analyzer.hf.space`

### ¿Qué puede salir mal?

| Problema | Causa probable | Solución |
|----------|---------------|----------|
| "Build failed" | Falta dependencia en `requirements.txt` | Revisar logs, añadir la dependencia |
| "Runtime error" | Error en el código Python | Revisar logs, corregir el código |
| Muy lento al cargar | Modelo grande descargándose | Normal la primera vez, después se cachea |
| "Out of memory" | Modelo demasiado grande para CPU | Usar un modelo más pequeño o pedir GPU |

> 🎓 El profesor lo hará en vivo. ¡Seguidlo paso a paso con vuestras cuentas de HuggingFace!

In [ ]:
# Código listo para desplegar en HuggingFace Spaces
# Copia esto en un archivo app.py

app_code = '''import gradio as gr
from transformers import pipeline

clasificador = pipeline("sentiment-analysis")

def analizar_sentimiento(texto: str) -> dict:
    resultado = clasificador(texto)[0]
    return {
        "POSITIVE 😊": resultado["score"] if resultado["label"] == "POSITIVE" else 1 - resultado["score"],
        "NEGATIVE 😞": resultado["score"] if resultado["label"] == "NEGATIVE" else 1 - resultado["score"],
    }

demo = gr.Interface(
    fn=analizar_sentimiento,
    inputs=gr.Textbox(label="Texto a analizar", placeholder="Escribe aquí...", lines=3),
    outputs=gr.Label(label="Sentimiento"),
    title="🎭 Análisis de Sentimiento",
    description="Escribe un texto y el modelo determinará si es positivo o negativo.",
    examples=[
        ["Me encanta este producto, funciona genial!"],
        ["Terrible experiencia, no lo recomiendo"],
        ["Está bien, funciona pero nada especial"],
    ]
)

demo.launch()
'''

requirements = "transformers\ntorch\n"

# Guardar archivos
with open("app.py", "w") as f:
    f.write(app_code)
with open("requirements.txt", "w") as f:
    f.write(requirements)

print("✅ Archivos creados:")
print("   app.py - Código de la aplicación Gradio")
print("   requirements.txt - Dependencias")
print("\nPara desplegar en HuggingFace Spaces:")
print("1. Ve a https://huggingface.co/new-space")
print("2. Sube app.py y requirements.txt")
print("3. ¡Listo!")

✅ Archivos creados:
   app.py - Código de la aplicación Gradio
   requirements.txt - Dependencias

Para desplegar en HuggingFace Spaces:
1. Ve a https://huggingface.co/new-space
2. Sube app.py y requirements.txt
3. ¡Listo!


---
# Bloque 5 — Recapitulación

Este último bloque tiene dos objetivos: primero, **conectar los puntos** de todo lo aprendido en las 4 sesiones para que veáis el panorama completo. Segundo, presentar el **proyecto final** — la oportunidad de aplicar todo en un entregable real.

## 🗺️ Recapitulación del Módulo

### El viaje completo: de cero a aplicación desplegada

En 4 sesiones habéis recorrido todo el camino desde "no sé Python" hasta "tengo una app web con IA desplegada en internet". Este es el mapa de lo que habéis aprendido:

### Sesión 1 — Fundamentos: Las herramientas y el idioma
- ✅ **Google Colab + Gemini** como entorno y copiloto de desarrollo
- ✅ **Python esencial** a vista de pájaro: variables, estructuras, funciones
- ✅ **Prompting para desarrollo**: cómo pedir código de forma efectiva
- ✅ **Git y GitHub**: control de versiones y colaboración

> *Resultado*: Podéis escribir y ejecutar Python, pedir código a la IA, y gestionar vuestro código con Git.

### Sesión 2 — Datos y Prompting: Las librerías y técnicas clave
- ✅ **NumPy y Pandas**: manipulación de datos numéricos y tabulares
- ✅ **Prompting avanzado**: Chain of Thought, Role Prompting, System Prompts
- ✅ **Desarrollo agéntico**: GitHub Copilot como segundo cerebro
- ✅ **Keras**: vuestra primera red neuronal (MNIST)

> *Resultado*: Podéis trabajar con datos, usar técnicas avanzadas de prompting, y entrenar un modelo básico.

### Sesión 3 — Ecosistema IA: Modelos y APIs
- ✅ **HuggingFace**: Hub, Pipeline, Transformers — miles de modelos a un import de distancia
- ✅ **APIs de LLMs**: consumir Gemini y otros modelos desde Python
- ✅ **LangChain y RAG**: chatbot sobre documentos propios
- ✅ **Modelos locales**: Ollama para ejecutar modelos sin internet

> *Resultado*: Podéis acceder a cualquier modelo de IA (local, API o HuggingFace) y construir sistemas avanzados como RAG.

### Sesión 4 — Interfaces y Deployment: Del código al producto
- ✅ **Python avanzado**: excepciones, POO, type hints — leer y generar código profesional
- ✅ **Gradio**: interfaces web para demos de modelos
- ✅ **Streamlit**: apps de datos completas
- ✅ **Deployment**: HuggingFace Spaces y Streamlit Cloud

> *Resultado*: Podéis crear interfaces web para vuestros modelos y desplegarlas online.

### La cadena completa

```
Sesión 1          Sesión 2          Sesión 3          Sesión 4
─────────         ─────────         ─────────         ─────────
Python +     →    Datos +      →    Modelos IA   →    Interfaz +
Prompting         Técnicas          APIs              Deployment
                  avanzadas         Ecosistema

                    = APLICACIÓN WEB CON IA DESPLEGADA
```

---
## 📚 Recursos adicionales

- [Gradio — Documentación](https://gradio.app/docs/)
- [Gradio — Quickstart](https://gradio.app/quickstart/)
- [Streamlit — Documentación](https://docs.streamlit.io/)
- [Streamlit — Cheat Sheet](https://docs.streamlit.io/develop/quick-reference/cheat-sheet)
- [HuggingFace Spaces — Docs](https://huggingface.co/docs/hub/spaces)
- [Streamlit Cloud — Deployment](https://docs.streamlit.io/deploy/streamlit-community-cloud)
- [Python Type Hints — Guía](https://docs.python.org/3/library/typing.html)
- [Google Docstring Style](https://google.github.io/styleguide/pyguide.html#38-comments-and-docstrings)

## 🎓 ¡Fin del Módulo!

### Lo que habéis aprendido — y lo que realmente importa

El objetivo de este módulo **nunca fue que memorizarais sintaxis de Python**. Eso lo hace la IA por vosotros. Lo que habéis aprendido es mucho más valioso:

- **Vocabulario técnico**: Sabéis qué son pipelines, transformers, embeddings, RAG, APIs, tokens, prompts... Esto os permite **comunicaros con equipos técnicos** y **pedir a la IA exactamente lo que necesitáis**.
- **Pensamiento de sistemas**: Entendéis cómo encajan las piezas — desde los datos hasta el modelo, desde la interfaz hasta el deployment.
- **Desarrollo asistido por IA**: No programáis solos — programáis **con un copiloto**. Y sabéis cómo dirigirlo (prompting, specs, iteración).
- **Autonomía**: Podéis **crear y desplegar** aplicaciones web con IA sin depender de un equipo de desarrollo. De la idea al producto en horas, no en meses.

### Lo que NO necesitáis (pero mucha gente cree que sí)

- ❌ Memorizar toda la sintaxis de Python
- ❌ Escribir todo el código desde cero
- ❌ Entender matemáticas avanzadas para usar modelos
- ❌ Ser "programadores senior" para crear apps útiles

### Lo que SÍ necesitáis (y ahora tenéis)

- ✅ Saber **qué existe** y cómo se llama
- ✅ Saber **qué pedirle a la IA** (prompting)
- ✅ Saber **evaluar** si lo que la IA genera es correcto
- ✅ Saber **desplegar** para que otros puedan usarlo

### Próximos pasos en el Máster

Todo lo que habéis aprendido aquí os servirá como base para los módulos que vienen:

| Módulo | Conexión con este módulo |
|--------|------------------------|
| **Machine Learning** | Usaréis NumPy, Pandas y las técnicas de prompting para explorar algoritmos |
| **Deep Learning** | Profundizaréis en redes neuronales (como la de Keras que visteis) |
| **NLP** | Los Transformers y HuggingFace pipelines serán vuestra herramienta diaria |
| **Computer Vision** | Aplicaréis pipelines de HuggingFace para imágenes |
| **TFM** | Crearéis una aplicación completa con interfaz y deployment |

> 🚀 **El mejor momento para empezar a construir con IA es ahora. Tenéis las herramientas, tenéis el conocimiento, tenéis un copiloto. ¡Mucho éxito en el Máster!**